# session

> Read, write, and build native Claude Code session transcripts

In [ ]:
#| default_exp session

## Session layout

Claude Code keeps conversations at `~/.claude/projects/<dir>/<session-id>.jsonl`. Each line contains one JSON record. The filename uses the conversation's first session id. Resumes and post-compaction restarts continue writing to that file even when `CLAUDE_CODE_SESSION_ID` changes.

`sess_dir` resolves the project path and replaces every non-alphanumeric character with `-`. This includes underscores. On macOS, `/tmp/foo` becomes `-private-tmp-foo`. `sess_file` locates a transcript by session id or unique prefix. `cur_sess` chooses the project's most recently modified transcript. Pass an explicit id when multiple conversations use the same project.

Conversation records have type `user` or `assistant`. Their main fields are `type`, `uuid`, `parentUuid`, `sessionId`, `timestamp`, and `message`. A tool result belongs to a `user` record with `tool_result` content blocks.

`load_sess` and `load_recs` read all records without changing the file. `sess_thread` follows `parentUuid` links backwards from the last eligible record to recover the active conversation. `rec_txt` extracts readable text from a record's message content.

## Building sessions

You can resume a transcript that you assembled yourself, including synthetic tool exchanges. `mk_rec` creates individual records. `msgs2recs` converts Anthropic-style message lists. `msgs2sess` also writes the transcript and returns its session id. These message-list conversions produce deterministic ids from the messages and a key.

Use `mk_tu` and `mk_tr` for tool-call and tool-result blocks. `tool_turn` builds a complete exchange. `prefix_tools` qualifies caller tool names to match Claude Code's transcript format.

`save_sess` replaces the entire target file and rebuilds its parent chain. `append_sess` links new records after the existing transcript. Inspect the target records and read these functions' documentation before writing. `llmsurgery` uses this module to find, search, curate, convert, and compact sessions.

A Claude Code transcript can serve as data, a reusable conversation template, or a starting history for a new session. You don't need a model call to create one. Write the records to a session file and run `claude --resume <session-id>`. Claude Code reads synthetic records in the same way as records from an earlier conversation. For example, you can give it worked examples of tool use without running those tools.

In [ ]:
#| export
import json, os, re, uuid
from datetime import datetime, timezone
from importlib.resources import files
from fastcore.utils import *
from fastcore.meta import delegates

In [ ]:
from fastcore.test import *
from fastclaude.core import astream
from aidialog.msg_parts import Msg, Text
from collections import Counter
import tempfile, shutil


## Where sessions live

`sess_dir` returns the storage directory for a project. It expands `~` and resolves symlinks before converting the path to a folder name. This matters on macOS: `/tmp` and `/var` point inside `/private`. A project at `/tmp/foo` uses the folder `~/.claude/projects/-private-tmp-foo`.

In [ ]:
#| export
SESSIONS = Path.home()/'.claude'/'projects'

def sess_dir(
    cwd=None, # Project directory; the current directory if None
):
    "The folder where Claude Code keeps session transcripts for the project at `cwd`"
    return SESSIONS/re.sub(r'[^a-zA-Z0-9]', '-', str(Path(cwd or '.').expanduser().resolve()))

In [ ]:
sess_dir()

Path('/Users/jhoward/.claude/projects/-Users-jhoward-aai-ws-fastclaude-nbs')

Underscores are replaced too, which is easy to get wrong when sanitizing by hand:

In [ ]:
test_eq(sess_dir('/a/b_c').name, '-a-b-c')
test_eq(sess_dir('~'), sess_dir(Path.home()))

`CLAUDE_CODE_SESSION_ID` identifies a Claude Code run. Claude Code passes it to child processes, including shell commands and MCP servers. The id matches the transcript filename during a conversation's first run. A resume or post-compaction restart can set a new id without creating a new transcript. Claude Code keeps appending to the file named after the conversation's first id.

`cur_sess` looks for that file by modification time. Appending a record updates the transcript's timestamp. The newest transcript usually belongs to the running conversation, including after a resume or restart. If the project has no transcripts, `cur_sess` returns `CLAUDE_CODE_SESSION_ID` instead.

This is a heuristic. Two conversations in one project take turns having the newest transcript. Pass an explicit `sid` if they can run concurrently. Clikernel resolves its host conversation once at worker startup, while the spawning conversation's transcript is the newest.

In [ ]:
#| export
def cur_sess(
    cwd=None, # Project directory; the current directory if None
):
    "The newest transcript's session id for `cwd`, or the advertised id if none exists"
    try: return max(sess_dir(cwd).glob('*.jsonl'), key=lambda p: p.stat().st_mtime).stem
    except ValueError: return os.environ.get('CLAUDE_CODE_SESSION_ID')

In [ ]:
cur_sess()

The test project has two transcripts with different modification times. `cur_sess` selects the newer file. For an empty project it falls back to the environment variable:

In [ ]:
tp = Path(tempfile.mkdtemp())
sd = sess_dir(tp)
sd.mkdir(parents=True)
for i,n in enumerate(['older','newer']):
    (sd/f'{n}.jsonl').touch()
    os.utime(sd/f'{n}.jsonl', (i,i))
test_eq(cur_sess(tp), 'newer')
test_eq(cur_sess(tempfile.mkdtemp()), os.environ.get('CLAUDE_CODE_SESSION_ID'))

`uniq_path` resolves a collection of candidate paths. It returns the sole distinct path, or `None` for an empty collection. More than one distinct path raises `ValueError` with the matching paths.

In [ ]:
#| export
def uniq_path(
    paths, # Candidate paths, e.g. from a glob
    ref, # The reference they were matched against, for the error message
):
    "The single path in `paths`, or None if there are none; raises if `ref` is ambiguous"
    res = sorted({str(o) for o in paths})
    if len(res)>1: raise ValueError(f'{ref!r} matches {len(res)} sessions:\n' + '\n'.join(res))
    return Path(res[0]) if res else None

In [ ]:
#| export
def sess_file(
    sid=None, # Session id or unique id prefix; `cur_sess(cwd)` if None
    cwd=None, # Project directory; the current directory, then all projects, if None
):
    "Path to the transcript of session `sid` for the project at `cwd`"
    sid = sid or cur_sess(cwd)
    p = sess_dir(cwd)/f'{sid}.jsonl'
    if p.exists(): return p
    root,pat = (sess_dir(cwd),f'{sid}*.jsonl') if cwd is not None else (SESSIONS,f'*/{sid}*.jsonl')
    return uniq_path(root.glob(pat), sid) or p

`sess_file()` uses `cur_sess` when you omit `sid`, with the same concurrency limitation. It first checks for an exact filename in the current project's folder.

Without an explicit `cwd`, the next step searches every project folder for the session id or prefix. Session ids are unique across projects. This lets you open a known session from a notebook kernel outside its project root. Supplying `cwd` restricts the search to that project.

In [ ]:
test_eq(sess_file('abc', '/a/b_c'), SESSIONS/'-a-b-c'/'abc.jsonl')

An id prefix, such as the first eight characters, works when it matches one transcript. An exact file in the selected project takes precedence over prefix matching. Ambiguous prefixes raise `ValueError`. With no match, `sess_file` returns the proposed path in the selected project even though the file doesn't exist.

In [ ]:
test_eq(sess_file('old', tp), sd/'older.jsonl')
for n in ['dup-a','dup-b']: (sd/f'{n}.jsonl').touch()
with expect_fail(contains='matches 2 sessions'): sess_file('dup', tp)
sess_file('new', tp)

Path('/Users/jhoward/.claude/projects/-private-var-folders-mv-nt5dfl8j0xbg7zkfw-sk-d8m0000gn-T-tmpxk461rxd/newer.jsonl')

In [ ]:
#| export
def load_recs(path):
    "Session records read directly from JSONL `path`"
    return dict2obj(Path(path).read_jsonl())

## Creating dummy data

The examples use captured Claude Code transcripts as fixtures. Each builder reuses its existing target. Normal notebook runs therefore read the same data without contacting Claude. To regenerate a fixture, remove its target file or directory before calling the builder.

In [ ]:
#| export
ant_data = Path(files('fastclaude')/'data'/'ant')

In [ ]:
def mk_ant_fixture_project(path=None):  # chkstyle: ignore-node
    "Create the minimal Claude project used by the ant fixtures"
    assert ant_data.is_dir(), f'ant fixtures ship as package data; missing {ant_data}'
    root = Path(path) if path else ant_data/'project'
    skill = root/'.claude/skills/ant-fixture/SKILL.md'
    if skill.exists(): return root
    skill.parent.mkdir(parents=True, exist_ok=True)
    skill.write_text("""---
name: ant-fixture
description: Report the fixed ant fixture fact when asked.
---

When invoked, report exactly: `The ant fixture skill is loaded.`
""")
    return root

The fixture project has one local skill with a fixed response. The package includes this project and the transcripts in `fastclaude/data/ant`. `importlib.resources` locates them independently of the working directory. Downstream packages, including `llmsurgery`, use the exported `ant_data` path to share these captures.

In [ ]:
mk_ant_fixture_project()

Path('/Users/jhoward/aai-ws/fastclaude/fastclaude/data/ant/project')

`mk_ant_source` uses `fastclaude.core.astream` to capture a short conversation. Claude loads the project skill, runs a Bash command, evaluates an expression through the persistent clikernel MCP server, and replies with fixed text. The transcript includes skill context, tool calls and results, and Claude Code's bookkeeping.

In [ ]:
async def mk_ant_source(path=None):  # chkstyle: ignore-node
    "Create the real Claude session used by the ant fixtures"
    path = Path(path) if path else ant_data/'source.jsonl'
    if path.exists(): return path
    root = mk_ant_fixture_project(path.parent/'project')
    cmd = shutil.which('clikernel-mcp')
    if not cmd: raise FileNotFoundError('clikernel-mcp')
    prompt = "Use the ant-fixture skill. Then use Bash to run `printf 'bash fixture\\n'`. Then use clikernel to evaluate `6*7`. After all tools finish, reply exactly: fixture complete."
    run = astream([Msg('user', [Text(prompt)])], model='haiku', system='Follow the requested tool sequence exactly.',
        cwd=root, native_tools=('Skill','Bash'), allowed=('Bash','mcp__clikernel__execute'),
        mcp_config=dict(clikernel=dict(type='stdio', command=cmd)), setting_sources=['project'], max_turns=8)
    sid = None
    async for m in run:
        if m.get('type')=='system' and m.get('subtype')=='init': sid = m.get('session_id')
    src = sess_file(sid, root)
    if not src.exists(): raise FileNotFoundError(src)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(src.read_bytes())
    return path


If `source.jsonl` is missing, `mk_ant_source` runs Claude and copies its transcript to `data/ant/source.jsonl`. This costs model tokens and requires `clikernel-mcp`. An existing file skips the model run. Regenerating the capture uses the installed Claude Code version.

In [ ]:
await mk_ant_source()

Path('/Users/jhoward/aai-ws/fastclaude/fastclaude/data/ant/source.jsonl')

The captured file contains conversation records and bookkeeping. Here's the count for each record type:

In [ ]:
source_path = ant_data/'source.jsonl'
source_recs = load_recs(source_path)
len(source_recs),Counter(source_recs.attrgot('type'))

(20,
 Counter({'assistant': 8,
          'user': 5,
          'attachment': 3,
          'queue-operation': 2,
          'ai-title': 1,
          'last-prompt': 1}))

`user` and `assistant` records contain chat messages. Attachments add context such as skill contents. Queue operations, titles, and prompt markers describe the session but aren't chat messages.

### The parent chain

In [ ]:
#| export
def sess_thread(
    recs, # Session records, e.g. from `load_sess`
):
    "The records on the active conversation chain, walking `parentUuid` back from the last record"
    recs = L(r for r in recs if r.get('type') in ('user','assistant','attachment','system') and r.get('uuid'))
    byid = {r.uuid:r for r in recs}
    cur,res,seen = recs[-1],[],set()
    while cur is not None:
        if cur.uuid in seen: raise ValueError(f'parentUuid cycle at record uuid {cur.uuid}')
        seen.add(cur.uuid)
        res.append(cur)
        cur = byid.get(cur.get('parentUuid'))
    return L(reversed(res))

Claude reconstructs the active conversation by following `parentUuid` backwards from its final linked record. `sess_thread` considers `user`, `assistant`, `attachment`, and `system` records with UUIDs. It excludes bookkeeping such as `custom-title`, even when those records have UUIDs. The returned records run from the oldest reachable parent to the final record.

In [ ]:
source_thread = sess_thread(source_recs)
len(source_recs),len(source_thread),Counter(source_thread.attrgot('type'))

(20, 16, Counter({'assistant': 8, 'user': 5, 'attachment': 3}))

In this fixture, four bookkeeping records don't belong to the active chain. The chain includes chat messages and context attachments. Each record after the first points to its predecessor:

In [ ]:
assert all(b.parentUuid==a.uuid for a,b in zip(source_thread,source_thread[1:]))

A missing parent link stops the walk. Earlier records become unreachable, as with an abandoned or malformed branch on resume. A corrupt file can also contain a cycle. `sess_thread` raises `ValueError` on a repeated UUID rather than looping forever. The writing functions reject duplicate UUIDs before writing:

In [ ]:
broken = L(obj2dict(r) for r in source_thread)
broken[-2]['parentUuid'] = None
test_eq(len(sess_thread(L(dict2obj(r) for r in broken))), 2)
cyc = L(dict2obj(obj2dict(r)) for r in source_thread)
cyc[-1]['parentUuid'] = cyc[-1]['uuid']
test_fail(lambda: sess_thread(cyc), contains='cycle')

In [ ]:
#| export
def _txts(o, skip=()):
    if isinstance(o, str): yield o
    elif isinstance(o, dict): yield from (t for k,v in o.items() if k not in skip for t in _txts(v, skip))
    elif is_listy(o): yield from (t for x in o for t in _txts(x, skip))

In [ ]:
#| export
def rec_txt(
    r, # A session record
):
    "Every readable string in `r`'s message content, joined, for finding records by text"
    return '\n'.join(_txts(obj2dict(r).get('message', {}).get('content', ''), skip=('type','id','tool_use_id','signature')))

## Writing a session

`mk_rec` wraps a message in a transcript record. The outer fields describe the session and the message's place in its history. We call these fields the record's envelope.

The function includes optional fields from real transcripts: `version`, `gitBranch`, `userType`, `entrypoint`, and `session_id`. It also supplies assistant API metadata. For a user tool result, `toolUseResult` repeats the result content for the transcript UI. Resume accepts fewer fields, but we don't know every effect of omitting them on Claude Code's model interaction. The defaults follow real transcripts.

By default, each call creates a fresh record UUID and timestamp. Rebuilding the same history therefore produces different records. For reproducible templates, provide fixed timestamps and deterministic ids. `canon` serializes values to canonical JSON. `stable_uuid` derives a UUID from a string. `fastllm_claude_code.core` uses these helpers for stable session and record ids.

In [ ]:
#| export
CC_VERSION = '2.1.223'

def canon(o):
    "Canonical compact JSON for `o`, key-sorted, for stable hashing"
    return json.dumps(o, sort_keys=True, separators=(',', ':'), ensure_ascii=False)

def stable_uuid(s):
    "A uuid deterministically derived from string `s`"
    return str(uuid.uuid5(uuid.NAMESPACE_URL, s))

`canon` sorts dictionary keys. Insertion order doesn't affect the resulting JSON. `stable_uuid` returns the same UUID for the same input string:

In [ ]:
test_eq(canon(dict(b=1, a=2)), canon(dict(a=2, b=1)))
test_eq(stable_uuid('x'), stable_uuid('x'))
assert stable_uuid('x') != stable_uuid('y')

In [ ]:
#| export
def _now(): return datetime.now(timezone.utc).strftime(r'%Y-%m-%dT%H:%M:%S.%f')[:-3]+'Z'

def _est_toks(o):
    "Rule-of-thumb token estimate for every string in `o`: words * 1.5"
    return int(sum(len(s.split()) for s in _txts(o))*1.5)

`mk_rec` accepts explicit record ids and timestamps when you need reproducible output. Its assistant usage fields contain token estimates rather than measurements from a model run.

In [ ]:
#| export
def mk_rec(
    role, # 'user' or 'assistant'
    content, # A string, or a list of content blocks
    cwd='.', # Project directory recorded in the envelope
    uid=None, # Record uuid; random if None
    ts=None, # ISO8601 timestamp; the current time if None
    model='claude-sonnet-4-6', # Recorded in assistant API metadata; None omits it, so resume uses the user's default
    input_toks=0, # `input_tokens` recorded in assistant usage, e.g. an estimate of the context so far
    **kwargs, # Extra or overriding envelope fields, e.g. `isCompactSummary=True`
):
    "A transcript record for one conversation message, ready for `save_sess`"
    uid = uid or str(uuid.uuid4())
    if isinstance(content, list):
        content = [{k:v for k,v in b.items() if k!='cache_control'} if isinstance(b, dict) else b for b in content]
        if role=='user' and content and all(isinstance(b, dict) and b.get('type')=='text' for b in content): content = ''.join(b['text'] for b in content)
    msg = dict(type='message', role=role, content=content)
    r = dict(type=role, uuid=uid, parentUuid=None, sessionId=None, timestamp=ts or _now(), cwd=str(Path(cwd).expanduser().resolve()),
        version=CC_VERSION, gitBranch='HEAD', isSidechain=False, userType='external', entrypoint='cli', session_id=None, message=msg)
    if role=='user' and isinstance(content, list) and (trs := [b for b in content if isinstance(b, dict) and b.get('type')=='tool_result']):
        c = trs[-1]['content']
        r['toolUseResult'] = c if isinstance(c, list) else [dict(type='text', text=c)]
    if role=='assistant':
        tu = isinstance(content, list) and any(isinstance(b, dict) and b.get('type')=='tool_use' for b in content)
        r['requestId'] = 'req_'+stable_uuid(f'{uid}:req').replace('-', '')[:24]
        usage = dict(input_tokens=input_toks, output_tokens=_est_toks(content), cache_creation_input_tokens=0, cache_read_input_tokens=0)
        msg.update(id='msg_'+stable_uuid(f'{uid}:msg').replace('-', '')[:24],
            stop_reason='tool_use' if tu else 'end_turn', stop_sequence=None, stop_details=None, usage=usage)
        if model: msg['model'] = model
    return dict(r, **kwargs)

In [ ]:
mk_rec('user', 'Hello!')

{'type': 'user',
 'uuid': '2e66a2c9-69be-4c7d-b63e-283dc68686c1',
 'parentUuid': None,
 'sessionId': None,
 'timestamp': '2026-09-09T07:07:44.795Z',
 'cwd': '/Users/jhoward/aai-ws/fastclaude/nbs',
 'version': '2.1.223',
 'gitBranch': 'HEAD',
 'isSidechain': False,
 'userType': 'external',
 'entrypoint': 'cli',
 'session_id': None,
 'message': {'type': 'message', 'role': 'user', 'content': 'Hello!'}}

`mk_rec` derives an assistant record's API ids from its record UUID. Any `tool_use` block sets `stop_reason` to `tool_use`. Without a tool call, the value is `end_turn`. Passing `model=None` omits the recorded model and lets resume use the user's default.

In [ ]:
tu = [dict(type='tool_use', id='toolu_01', name='probe', input={})]
r = mk_rec('assistant', tu, uid=stable_uuid('demo'), ts='2026-01-01T00:00:00.000Z')
test_eq(r['message']['stop_reason'], 'tool_use')
test_eq(r, mk_rec('assistant', tu, uid=stable_uuid('demo'), ts='2026-01-01T00:00:00.000Z'))
assert 'requestId' in r and 'requestId' not in mk_rec('user', 'hi')
test_eq(r['message']['usage']['output_tokens'], _est_toks(tu))
assert 'model' not in mk_rec('assistant', tu, model=None)['message']
test_eq(mk_rec('user', 'hi', cwd='~')['cwd'], str(Path.home()))

Anthropic API messages need two adjustments for transcript storage. `mk_rec` removes `cache_control` from content blocks. It also joins text-only user blocks into one string, matching Claude Code's recorded user messages. Tool-result blocks remain blocks:

In [ ]:
cb = [dict(type='text', text='Hi ', cache_control=dict(type='ephemeral')), dict(type='text', text='there.')]
test_eq(mk_rec('user', cb)['message']['content'], 'Hi there.')
test_eq(mk_rec('user', [dict(type='text', text='hi')])['message']['content'], 'hi')
tr = mk_rec('user', [dict(type='tool_result', tool_use_id='t1', content='ok', cache_control=dict(type='ephemeral'))])
test_eq(tr['message']['content'], [dict(type='tool_result', tool_use_id='t1', content='ok')])

`save_sess` writes records in the supplied order and returns the session id. It replaces any existing target file. For each record with a UUID, it sets `sessionId` and links `parentUuid` to the preceding UUID.

This creates a linear history. It does not preserve existing branches. Write the records yourself if you need to retain their original parent links. Records without a `uuid`, such as `last-prompt` and `mode`, keep their contents and position without joining the parent chain.

In [ ]:
#| export
def _chain(recs, sid, prev, ts, used=()):
    "Chain `recs` for session `sid` after uuid `prev`, refusing duplicate uuids (a cycle on disk otherwise)"
    seen = set(used)
    for r in recs:
        if 'uuid' not in r: continue   # bookkeeping records (last-prompt, mode, ...) ride along unchained
        if (u := r['uuid']) in seen: raise ValueError(f'duplicate record uuid {u}')
        seen.add(u)
        r['sessionId'],r['parentUuid'],prev = sid,prev,u
        if ts: r['timestamp'] = _now() if ts is True else ts
        if 'session_id' in r: r['session_id'] = sid

def save_sess(
    recs, # Records in conversation order, e.g. from `mk_rec`
    sid=None, # Session id; a fresh uuid if None
    cwd=None, # Project directory; the current directory if None
    ts=None, # If given, stamp every record's timestamp: True for the current time, or an ISO8601 string
):
    "Chain `recs`, write them as session `sid` for the project at `cwd`, and return `sid`"
    sid = sid or str(uuid.uuid4())
    _chain(recs, sid, None, ts)
    f = sess_file(sid, cwd or '.')
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(''.join(xdumps(r)+'\n' for r in recs))
    return sid

In [ ]:
#| export
def append_sess(
    recs, # Records to append, e.g. a munged template round
    sid=None, # Session to append to; `cur_sess()` if None
    cwd=None, # Project directory; the current directory if None
    ts=None, # If given, stamp each appended record's timestamp: True for the current time, or an ISO8601 string
):
    "Chain `recs` onto the tail of session `sid` and append them to its transcript, returning `sid`"
    sid = sid or cur_sess(cwd)
    old = [r['uuid'] for r in load_sess(sid, cwd) if 'uuid' in r]
    _chain(recs, sid, last(old), ts, old)
    with sess_file(sid, cwd).open('a') as f: f.writelines(xdumps(r)+'\n' for r in recs)
    return sid

`append_sess` adds records after the transcript's last UUID. It doesn't rewrite existing bytes or alter existing compaction records and sidechains. For example, llmdojo's `claudedojo -r` uses it to add a worked round after compaction. Both writers update the supplied record dictionaries when assigning the session id and parent links.

`msgs2recs` accepts Anthropic-style dictionaries with `role` and `content`. These include the output of `claude_mk_msg` and fastllm's `denorm_msgs`. It produces one transcript record per message, ready for `save_sess`.

The message content, position, and `key` determine each record id. With unchanged arguments, rebuilding the records produces byte-identical JSON. Changing the timestamp, project path, or other envelope arguments changes that output even if the ids stay the same.

In [ ]:
#| export
def msgs2recs(
    msgs, # Anthropic-style messages: dicts with `role` and `content`
    key='', # Salt: the same messages and key give the same ids
    cwd='.', # Project directory recorded in the envelopes
    ts='2026-01-01T00:00:00.000Z', # Timestamp for every record
    model='claude-sonnet-4-6', # Recorded in assistant API metadata; None omits it, so resume uses the user's default
    **kwargs, # Extra envelope fields for every record, e.g. `entrypoint`
):
    "Deterministic transcript records for `msgs`, one record per message"
    out,tot = [],0
    for i,m in enumerate(msgs):
        out.append(mk_rec(m['role'], m['content'], cwd=cwd, uid=stable_uuid(f'{key}:{i}:{canon(m)}'), ts=ts, model=model, input_toks=tot, **kwargs))
        tot += _est_toks(m['content'])
    return out

Here, changing `key` changes the record ids. The records retain the original roles. Assistant usage estimates include the preceding messages in `input_tokens`:

In [ ]:
den = [dict(role='user', content='Ping?'), dict(role='assistant', content=[dict(type='text', text='Pong.')])]
r1,r2 = msgs2recs(den, 'k'),msgs2recs(den, 'k')
test_eq(canon(r1), canon(r2))
test_ne(r1[0]['uuid'], msgs2recs(den, 'other')[0]['uuid'])
test_eq([r['type'] for r in r1], ['user','assistant'])
test_eq(r1[1]['message']['usage'], dict(input_tokens=1, output_tokens=3, cache_creation_input_tokens=0, cache_read_input_tokens=0))

`msgs2sess` converts a message list and writes its transcript in one call. It returns the id to pass to resume. The messages, `key`, and any `extra` records determine the session id. Repeating a call with the same history rewrites the same file instead of creating another transcript. This is useful for stateless callers that reconstruct their history on each request.

In [ ]:
#| export
@delegates(msgs2recs)
def msgs2sess(
    msgs, # Anthropic-style messages: dicts with `role` and `content`
    key='', # Salt: the same messages and key give the same session id
    cwd='.', # Project directory the session is filed under
    extra=None, # Records appended after the messages, e.g. from `mk_deferred`
    **kwargs
):
    "Write `msgs` as a resumable session with a content-stable id, returning the sid"
    sid = stable_uuid(f'{key}:{canon(msgs)}:{canon(extra or [])}')
    return save_sess(msgs2recs(msgs, key=sid, cwd=cwd, **kwargs)+list(extra or []), sid, cwd)

In [ ]:
tp2 = Path(tempfile.mkdtemp())
psid = msgs2sess(den, key='pingpong', cwd=tp2)
test_eq(msgs2sess(den, key='pingpong', cwd=tp2), psid)
back = load_recs(sess_file(psid, tp2))
test_eq(back[-1].message.content[0].text, 'Pong.')
test_eq(sess_thread(back).attrgot('uuid'), back.attrgot('uuid'))

### Forking

In [ ]:
#| export
def sess_id(recs):
    "The session id in transcript records `recs`"
    return first(r.get('sessionId') for r in recs if r.get('sessionId'))

`fork_session` copies session records into a new transcript without running a model. It assigns a fresh session id and fresh record UUIDs, then rebuilds the parent links. Each copied record has a `forkedFrom` field identifying its source session and record.

The copy excludes sidechains. It follows links through progress records but omits those records from the output. Only the last copied conversation record gets a new timestamp. Earlier timestamps remain unchanged. This lets resume find the final record by recency.

A `custom-title` record names the fork. Without an explicit title, the function uses the source's last custom title, then its last AI title, then its first user prompt line. It adds ` (fork)` to that name. `up_to` limits the copy to a chosen record, inclusive.


In [ ]:
#| export
def _fork_title(recs):
    "The source's last custom or AI title, else its first user prompt line"
    for k in ('customTitle','aiTitle'):
        if t := last((r[k] for r in recs if r.get(k)), None): return t
    return first((rec_txt(r).partition('\n')[0] for r in recs if r.get('type')=='user'), 'Forked session')

def fork_session(
    sid=None, # Session id or unique prefix to fork; the current session if None
    cwd=None, # Project directory; the current directory if None
    up_to=None, # Record uuid to cut at (inclusive); the whole transcript if None
    title=None, # Fork title; the source's `_fork_title` plus ' (fork)' if None
):
    "Fork a session natively: fresh session id and record uuids, rebuilt parent chain; returns the new sid"
    recs = load_recs(sess_file(sid, cwd))
    ssid = sess_id(recs)
    keep = [r for r in recs if r.get('type') in ('user','assistant','attachment','system','progress')
        and isinstance(r.get('uuid'), str) and not r.get('isSidechain')]
    if up_to:
        idx = first((i for i,r in enumerate(keep) if r['uuid']==up_to), None)
        if idx is None: raise ValueError(f'uuid not in transcript: {up_to}')
        keep = keep[:idx+1]
    ids = {r['uuid']: str(uuid.uuid4()) for r in keep}
    by_id = {r['uuid']: r for r in keep}
    fsid,now = str(uuid.uuid4()),_now()
    writable = [r for r in keep if r.get('type')!='progress']
    out = []
    for i,r in enumerate(writable):
        pid = r.get('parentUuid')
        while pid and by_id.get(pid, {}).get('type')=='progress': pid = by_id[pid].get('parentUuid')
        f = {**r, 'uuid': ids[r['uuid']], 'parentUuid': ids.get(pid), 'sessionId': fsid, 'isSidechain': False}
        f['timestamp'] = now if i==len(writable)-1 else r.get('timestamp', now)
        f['logicalParentUuid'] = ids.get(lp, lp) if (lp := r.get('logicalParentUuid')) else None
        f['forkedFrom'] = dict(sessionId=ssid, messageUuid=r['uuid'])
        for k in ('teamName','agentName','slug','sourceToolAssistantUUID'): f.pop(k, None)
        out.append(f)
    reps = [x for r in recs if r.get('type')=='content-replacement' and r.get('sessionId')==ssid for x in (r.get('replacements') or [])]
    if reps: out.append(dict(type='content-replacement', sessionId=fsid, replacements=reps, uuid=str(uuid.uuid4()), timestamp=now))
    out.append(dict(type='custom-title', sessionId=fsid, customTitle=title or f'{_fork_title(recs)} (fork)', uuid=str(uuid.uuid4()), timestamp=now))
    p = sess_dir(cwd)/f'{fsid}.jsonl'
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(''.join(json.dumps(obj2dict(r), separators=(',',':'))+'\n' for r in out))
    return fsid

In [ ]:
async def mk_ant_fork(path=None, source=None):  # chkstyle: ignore-node
    "Create the native Claude fork used by the ant fixtures"
    path,source = Path(path) if path else ant_data/'fork.jsonl',Path(source) if source else ant_data/'source.jsonl'
    if path.exists(): return path
    await mk_ant_source(source)
    root = mk_ant_fixture_project(source.parent/'project')
    sid = sess_id(load_recs(source))
    live = sess_file(sid, root)
    live.parent.mkdir(parents=True, exist_ok=True)
    live.write_bytes(source.read_bytes())
    src = sess_file(fork_session(sid, root, title='ant fixture fork'), root)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(src.read_bytes())
    return path


`mk_ant_fork` applies `fork_session` to the source capture and saves the result as `fork.jsonl` beside it. An existing fork file skips that work. Remove it to regenerate the fork.

The packaged fork is already available. The fixture-generation call is excluded from automated notebook runs:

In [ ]:
#| eval: false
await mk_ant_fork()

Path('data/ant/fork.jsonl')

Read the source and fork to compare their session ids and record counts:

In [ ]:
fork_path = ant_data/'fork.jsonl'
fork_recs = load_recs(fork_path)
source_sid,fork_sid = sess_id(source_recs),sess_id(fork_recs)
source_sid,fork_sid,len(source_recs),len(fork_recs),Counter(fork_recs.attrgot('type'))

('e12b4bd3-4f36-4dda-873d-2ff25d1f1044',
 '5d2b9564-f7f8-48c3-b611-f33fbe39c577',
 20,
 17,
 Counter({'assistant': 8, 'user': 5, 'attachment': 3, 'custom-title': 1}))

The fork has a new session id and a `custom-title` record. It contains the copied conversation but excludes the source's unlinked bookkeeping.

In [ ]:
source_thread,fork_thread = sess_thread(source_recs),sess_thread(fork_recs)
len(source_thread),len(fork_thread),Counter(source_thread.attrgot('type')),Counter(fork_thread.attrgot('type'))

(16,
 16,
 Counter({'assistant': 8, 'user': 5, 'attachment': 3}),
 Counter({'assistant': 8, 'user': 5, 'attachment': 3}))

The two chains contain the same sequence of record types. The fork's new UUIDs form a parent chain under its new session id:

In [ ]:
test_ne(source_sid, fork_sid)
test_eq(source_thread.attrgot('type'), fork_thread.attrgot('type'))
test_eq(set(fork_thread.attrgot('sessionId')), {fork_sid})
test_ne(source_thread.attrgot('uuid'), fork_thread.attrgot('uuid'))
test_eq(fork_thread.attrgot('parentUuid'), [None,*fork_thread.attrgot('uuid')[:-1]])
[(a.uuid,b.uuid) for a,b in zip(source_thread[:3],fork_thread[:3])]

[('f711b566-2404-41a5-a1b6-36f99d52e391',
  '58718dc0-80ed-4298-befc-c933e7913a94'),
 ('071ddef8-7dfb-42dc-b32c-38506751d50e',
  '1e84ec59-9412-44c1-8f12-16aeef9a201c'),
 ('05dd6db4-a8a1-422a-9617-ea7b19a00873',
  '2773c527-c1a1-4816-9a3f-435a9cde9f45')]

Changing the record ids and session fields doesn't change the readable conversation. Downstream compaction examples use this same fork fixture:

In [ ]:
test_eq([rec_txt(r) for r in source_thread], [rec_txt(r) for r in fork_thread])

Forking the source into a scratch project reproduces the fixture's record field sets and readable conversation. The example also supplies an explicit title:

In [ ]:
fproj = Path(tempfile.mkdtemp())
live = sess_file(source_sid, fproj)
live.parent.mkdir(parents=True, exist_ok=True)
live.write_bytes((ant_data/'source.jsonl').read_bytes())
myfork = fork_session(source_sid, fproj, title='ant fixture fork')
mrecs = load_recs(sess_file(myfork, fproj))
test_ne(myfork, source_sid)
test_eq([sorted(r.keys()) for r in mrecs], [sorted(r.keys()) for r in fork_recs])
test_eq([rec_txt(r) for r in sess_thread(mrecs)], [rec_txt(r) for r in source_thread])
first(r for r in mrecs if r.get('type')=='custom-title')['customTitle']


'ant fixture fork'

The next fork stops at the third conversation record. After checking its length, we remove the scratch project and its transcripts:

In [ ]:
cut = fork_session(source_sid, fproj, up_to=source_thread[2]['uuid'])
test_eq(len(sess_thread(load_recs(sess_file(cut, fproj)))), 3)
shutil.rmtree(sess_dir(fproj))
shutil.rmtree(fproj)

## Synthetic tool calls

An assistant requests a tool with a `tool_use` block. The following user record answers with a `tool_result` block. Both blocks refer to the same tool-call id. `mk_tr` takes the block from `mk_tu` and copies that id.

`tool_turn` builds four records: the user's request, the assistant's tool call, the user's tool result, and the assistant's closing reply.

In [ ]:
#| export
def mk_tu(
    name, # Tool name, as the transcript records it
    input=None, # Tool arguments
    tid=None, # tool_use id; random if None
):
    "A `tool_use` content block"
    return dict(type='tool_use', id=tid or 'toolu_'+uuid.uuid4().hex[:24], name=name, input=input or {})

def mk_tr(
    tu, # The `tool_use` block being answered
    content, # The tool's output
):
    "The `tool_result` content block answering `tu`"
    return dict(type='tool_result', tool_use_id=tu['id'], content=content)

def tool_turn(
    prompt, # The user request
    name, # Tool name
    input, # Tool arguments
    output, # Tool result
    answer, # The assistant's closing text
    **kwargs, # Passed to each `mk_rec`, e.g. `cwd`
):
    "A complete synthetic tool-use turn, as four records ready for `save_sess`"
    tu = mk_tu(name, input)
    return [mk_rec('user', prompt, **kwargs), mk_rec('assistant', [tu], **kwargs),
        mk_rec('user', [mk_tr(tu, output)], **kwargs), mk_rec('assistant', [dict(type='text', text=answer)], **kwargs)]

We'll use one of these four-record exchanges for the sample session below.

A `PreToolUse` hook can return `permissionDecision: "defer"`. Claude Code then ends the turn and stores the pending call in a `hook_deferred_tool` attachment. On resume, it re-invokes that call.

You can construct the same state without running the hook. Write a conversation ending with an assistant `tool_use` block, then append the corresponding `mk_deferred` record. No separate process state is necessary for replay. `fastclaude.core` uses this technique to continue a history ending in a tool result.

In [ ]:
#| export
def mk_deferred(
    tu, # The pending `tool_use` block dict
    cwd='.', # Project directory recorded in the envelope
    uid=None, # Record uuid; random if None
    ts='2026-01-01T00:00:00.000Z', # Timestamp recorded in the envelope
):
    "A `hook_deferred_tool` attachment record marking `tu` as deferred, ready for `save_sess`"
    return dict(type='attachment', uuid=uid or str(uuid.uuid4()), parentUuid=None, sessionId=None, isSidechain=False,
        timestamp=ts, userType='external', cwd=str(Path(cwd).expanduser().resolve()), version=CC_VERSION, gitBranch='HEAD',
        attachment=dict(type='hook_deferred_tool', toolUseID=tu['id'], toolName=tu['name'], toolInput=tu.get('input', {}),
            hookName='settings', hookEvent='PreToolUse', permissionMode='default'))

In [ ]:
dtu = mk_tu('mcp__probe__flux_meter', tid='toolu_01')
datt = mk_deferred(dtu, uid=stable_uuid('datt'))
test_eq(datt['attachment'], dict(type='hook_deferred_tool', toolUseID='toolu_01', toolName='mcp__probe__flux_meter',
    toolInput={}, hookName='settings', hookEvent='PreToolUse', permissionMode='default'))
datt['attachment']

{'type': 'hook_deferred_tool',
 'toolUseID': 'toolu_01',
 'toolName': 'mcp__probe__flux_meter',
 'toolInput': {},
 'hookName': 'settings',
 'hookEvent': 'PreToolUse',
 'permissionMode': 'default'}

## A sample session

Let's give a synthetic tool result a fact that Claude couldn't know otherwise: the flux is 41.7 kilofinches. No tool actually ran. We save the four-record exchange under a scratch project directory.

In [ ]:
proj = Path(tempfile.mkdtemp())
sample = tool_turn('Measure the flux please.', 'flux_meter', {}, 'flux: 41.7 kilofinches',
    'The flux reading is 41.7 kilofinches.', cwd=proj)
sid = save_sess(sample, cwd=proj)
sid

'029b4367-3c8f-415a-9ec4-b6fc86c57abd'

A loaded transcript can include records without UUIDs, such as `last-prompt`, `mode`, and `file-history-snapshot`. When you write it back with `save_sess`, these records stay unchanged in their original positions. The parent chain skips them:

In [ ]:
bk = dict(type='last-prompt', prompt='Measure the flux please.')
bsid = save_sess([sample[0], bk, *sample[1:]], cwd=proj)
back = load_recs(sess_file(bsid, proj))
test_eq(back[1], bk)
test_eq(back[2].parentUuid, back[0].uuid)
bsid

'85157b02-9b56-4ba4-858e-407d2c43804c'

## Reading a session

`load_sess` finds the transcript with `sess_file` and reads it with `load_recs`. The result uses `dict2obj` dictionaries, which support attribute access to fields.

In [ ]:
#| export
def load_sess(
    sid=None, # Session id; the current session if None
    cwd=None, # Project directory; the current directory if None
):
    "The records of session `sid`, as an `L` of attribute-access dicts"
    return load_recs(sess_file(sid, cwd))

The saved session has four records. Its tool result refers to the assistant's tool-call id:

In [ ]:
back = load_sess(sid, proj)
test_eq(len(back), 4)
test_eq(back[-1].message.content[0].text, 'The flux reading is 41.7 kilofinches.')
test_eq(back[2].message.content[0].tool_use_id, back[1].message.content[0].id)

The required fields for a resumable conversation record are `type`, `uuid`, `parentUuid`, `sessionId`, `timestamp`, and `message`. Without timestamps, Claude Code doesn't find the session on resume. The other envelope fields provide optional bookkeeping.

`message` follows the Anthropic API format: a `role` and `content`. Content can be a string or a list of blocks, including `text`, `tool_use`, `tool_result`, and `thinking`.

Real assistant records also include API metadata such as `requestId`, `message.id`, `model`, and usage. Resume does not require these fields. Synthetic histories also work without `thinking` blocks.

Reading the file back checks its contents. Resuming it checks whether Claude Code accepts the synthetic history. This optional example asks Claude for the flux reading from the fabricated tool result. It spends tokens and is excluded from automated notebook runs.

In [ ]:
#| eval: false
opts = ClaudeAgentOptions(resume=sid, cwd=str(proj), model='haiku')
async for m in query(prompt='What is the flux reading? Reply with only the value.', options=opts):
    if isinstance(m, ResultMessage): print(m.result)

41.7 kilofinches


## Qualified tool names

Claude Code records caller-hosted tools as `mcp__<server>__<name>`. Earlier calls in a resumed history must use those qualified names to match the available tools.

`prefix_tools` returns messages with a prefix added to unqualified `tool_use` names. It leaves names beginning with `mcp__` unchanged. Pass native tool names in `skip` to keep them unqualified. For example, Claude Code's `WebSearch` uses its bare name. The input messages don't change.

In [ ]:
#| export
def prefix_tools(
    msgs, # Anthropic-style messages
    prefix, # Stub name prefix, e.g. 'mcp__probe__'
    skip=(), # Native tool names to leave unqualified, e.g. 'WebSearch'
):
    "Copy of `msgs` with client `tool_use` names qualified by `prefix`"
    def _fix(b):
        if not (isinstance(b, dict) and b.get('type')=='tool_use'): return b
        nm = b.get('name','')
        if not nm or nm.startswith('mcp__') or nm in skip: return b
        return dict(b, name=prefix+nm)
    return [dict(m, content=[_fix(b) for b in m['content']]) if isinstance(m.get('content'), list) else m for m in msgs]

In [ ]:
tmsgs = [dict(role='assistant', content=[dict(type='text', text='Checking.'), dict(type='tool_use', id='t1', name='flux_meter', input={})]),
    dict(role='assistant', content=[dict(type='tool_use', id='t2', name='WebSearch', input=dict(query='flux'))])]
pref = prefix_tools(tmsgs, 'mcp__probe__', skip=['WebSearch'])
test_eq([b['name'] for m in pref for b in m['content'] if b['type']=='tool_use'], ['mcp__probe__flux_meter','WebSearch'])
test_eq(tmsgs[0]['content'][1]['name'], 'flux_meter')

## Cleanup

Remove the sample from `~/.claude/projects`, along with the scratch project.

In [ ]:
shutil.rmtree(sess_dir(proj))
shutil.rmtree(proj)

In [ ]:
#| hide
#| eval: false
import nbdev
nbdev.nbdev_export()